# Phase 2: PD Model Feature Selection

**Author:** Hoang
**Date:** 2026-05-07
**Inputs:** `artifacts/phase2_rerun_v2/modeling_abt.parquet`
**Outputs:** `artifacts/phase2_rerun_v2/{stage1, stage2, stage3, final_model.pkl}`

Three-stage feature selection on the 800/day production wide ABT (post Phase 1.5
feature factory, post `synth_credit_score_*` -> `synth_bureau_score_*` rename):

1. **Stage 1** — univariate prescreening (sparsity, variance, signal, stability)
2. **Stage 2** — Lasso (L1-penalised logistic regression, 5-fold CV)
3. **Stage 3** — statsmodels logit refinement + VIF + p-value filter

In [1]:
# Set up sys.path so 'from src import ...' works from a notebook in notebooks/
import sys
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import os
import time

import joblib
import numpy as np
import pandas as pd

from src.governance import (
    filter_pd_eligible, validate_no_score_columns,
    validate_no_loan_term_in_pd, get_meta_columns, summarise_eligible_pool,
)
from src.modeling import (
    run_stage1_prescreening, run_lasso_selection,
    fit_logit_statsmodels, compute_vif,
)
from src.evaluation import (
    compute_gini, compute_ks, compute_brier, compute_calibration_metrics,
)

np.random.seed(42)

T0 = time.time()

In [2]:
ABT_PATH     = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_abt.parquet"
CATALOG_PATH = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_feature_catalog.csv"
OUTPUT_DIR   = REPO_ROOT / "artifacts/phase2_rerun_v2"
CACHE_DIR    = REPO_ROOT / "artifacts/notebook_cache"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"ABT_PATH    : {ABT_PATH}")
print(f"CATALOG_PATH: {CATALOG_PATH}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"CACHE_DIR   : {CACHE_DIR}")

ABT_PATH    : E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\phase2_rerun_v2\modeling_abt.parquet
CATALOG_PATH: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\phase2_rerun_v2\modeling_feature_catalog.csv
OUTPUT_DIR  : E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\phase2_rerun_v2
CACHE_DIR   : E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\notebook_cache


## Dry-Run Validation

Load catalog, filter PD-eligible features, then load only those columns + meta
from the ABT. Verify shapes, DRs, score-column absence, and loan-term exclusion.

In [3]:
catalog = pd.read_csv(CATALOG_PATH)
print(f"Catalog: {len(catalog):,} rows, {len(catalog.columns)} cols")

pd_eligible = filter_pd_eligible(catalog)
print(f"PD-eligible features: {len(pd_eligible)}")

# Per-family breakdown of the eligible pool
elig_summary = summarise_eligible_pool(catalog)
print("\nPD-eligible by family:")
print(elig_summary.to_string(index=False))

columns_to_load = pd_eligible + get_meta_columns()
df = pd.read_parquet(ABT_PATH, columns=columns_to_load)
print(f"\nLoaded shape : {df.shape}")
print(f"Memory       : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"Train rows   : {(df['split']=='train').sum():,}")
print(f"OOT rows     : {(df['split']=='oot').sum():,}")
print(f"Train DR     : {df.loc[df['split']=='train', 'default_flag_12m'].mean():.4%}")
print(f"OOT DR       : {df.loc[df['split']=='oot', 'default_flag_12m'].mean():.4%}")

f6d_count = sum(f.startswith('random_') for f in pd_eligible)
print(f"F6D pure-random in PD-eligible pool: {f6d_count} "
      f"({100*f6d_count/len(pd_eligible):.1f}%)")

# Defensive checks
assert validate_no_score_columns(df), "score/scorem leak detected!"
assert validate_no_loan_term_in_pd(df, catalog), "loan-term leak in PD pool!"
print("\nDefensive checks: PASS (no score/scorem, no loan-term in PD)")

Catalog: 2,236 rows, 14 cols
PD-eligible features: 435

PD-eligible by family:
feature_family  pd_eligible_count
           F6E                200
           F6D                100
           F6B                 47
           F5A                 37
           F6A                 25
  ORIGINAL_APP                 11
            F4                  6
           F6C                  5
            F3                  4



Loaded shape : (534314, 439)
Memory       : 1.02 GB
Train rows   : 389,525
OOT rows     : 144,789
Train DR     : 1.6330%
OOT DR       : 0.8474%
F6D pure-random in PD-eligible pool: 100 (23.0%)

Defensive checks: PASS (no score/scorem, no loan-term in PD)


## Stage 1 — Univariate Prescreening

Filter rules (per `src.modeling.run_stage1_prescreening`):
- **Sparsity:** `pct_nan <= 0.5`
- **Variance:** train stddev > 0
- **Signal:** train_gini >= 0.02
- **Stability:** `|train_gini - oot_gini| / max(train_gini, eps) <= 0.20`

Cached at `artifacts/notebook_cache/stage1_results.pkl` for re-runs.

In [4]:
cache_path = CACHE_DIR / "stage1_results.pkl"
if cache_path.exists():
    stage1 = joblib.load(cache_path)
    print(f"Loaded cached Stage 1 from {cache_path}")
else:
    t0 = time.time()
    stage1 = run_stage1_prescreening(
        df, target_col="default_flag_12m", features=pd_eligible
    )
    print(f"Stage 1 wall: {time.time() - t0:.1f}s")
    joblib.dump(stage1, cache_path)

print(f"Stage 1 survivors: {len(stage1)}")

Loaded cached Stage 1 from E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\notebook_cache\stage1_results.pkl
Stage 1 survivors: 120


In [5]:
print("Top 20 Stage 1 survivors by train Gini:")
print(stage1.head(20).to_string(index=False))

Top 20 Stage 1 survivors by train Gini:
                    feature  pct_nan           std  train_gini  oot_gini  gini_drop_pct  pass_sparse  pass_variance  pass_signal  pass_stability  survives
           app_nom_job_code      0.0      0.951194      0.3719    0.4328         0.1638         True           True         True            True      True
       mean_age_by_job_code      0.0      3.390114      0.3719    0.4328         0.1638         True           True         True            True      True
mean_n_children_by_job_code      0.0      0.113199      0.3719    0.4328         0.1638         True           True         True            True      True
                    act_age      0.0      6.243303      0.3406    0.3827         0.1237         True           True         True            True      True
                    log_age      0.0      0.106114      0.3406    0.3827         0.1236         True           True         True            True      True
                  act_age_z   

## Stage 2 — Lasso Selection

`LogisticRegressionCV` with L1 penalty, SAGA solver, 5-fold CV, 10 candidate
inverse regularisation strengths. Inputs are standardised; NaNs filled with
train medians. Returns features with non-zero coefficient at the CV-best `C`.

In [6]:
cache_path = CACHE_DIR / "stage2_results.pkl"
if cache_path.exists():
    stage2 = joblib.load(cache_path)
    print(f"Loaded cached Stage 2 from {cache_path}")
else:
    t0 = time.time()
    stage2 = run_lasso_selection(
        df, features=stage1["feature"].tolist(),
        target_col="default_flag_12m",
        cv=5, n_cs=10, random_state=42,
    )
    print(f"Stage 2 wall: {time.time() - t0:.1f}s")
    joblib.dump(stage2, cache_path)

print(f"Stage 2 Lasso survivors: {len(stage2)}")

Loaded cached Stage 2 from E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\notebook_cache\stage2_results.pkl
Stage 2 Lasso survivors: 64


In [7]:
print("Stage 2 survivors:")
for f in stage2:
    g = stage1.loc[stage1['feature']==f, 'train_gini']
    g_val = float(g.iloc[0]) if len(g) else float('nan')
    print(f"  {f:<45}  train_gini={g_val:.4f}")

Stage 2 survivors:
  app_nom_job_code                               train_gini=0.3719
  act_age_log1p                                  train_gini=0.3406
  synth_thin_file_flag                           train_gini=0.3373
  synth_age_of_newest                            train_gini=0.2856
  synth_newest_account_months                    train_gini=0.2535
  app_nom_marital_status                         train_gini=0.2448
  synth_avg_account_age                          train_gini=0.2423
  synth_oldest_mortgage_months                   train_gini=0.2384
  synth_seg_app_nom_job_code_top1                train_gini=0.2367
  synth_credit_age_years                         train_gini=0.2314
  age_x_n_children                               train_gini=0.2280
  synth_int_age_x_nchildren                      train_gini=0.2280
  marital_x_n_children                           train_gini=0.2220
  synth_int_inc_x_nchildren                      train_gini=0.2193
  synth_oldest_tradeline_v1                

## Stage 3 — Refinement (statsmodels Logit + VIF + p-value)

Fit unpenalised logit on Stage 2 survivors, inspect VIF, drop high-VIF features
(threshold 10), then drop p-values >= 0.05 and refit.

In [8]:
final_features = list(stage2)

vif_df = compute_vif(df, final_features)
print(f"VIF (top 15 by VIF):")
print(vif_df.head(15).to_string(index=False))

# Drop features with VIF > 10 (multicollinear)
high_vif = vif_df.loc[vif_df['vif'] > 10, 'feature'].tolist()
if high_vif:
    print(f"\nDropping {len(high_vif)} high-VIF features:")
    for f in high_vif:
        print(f"  - {f}")
    final_features = [f for f in final_features if f not in high_vif]
else:
    print("\nNo high-VIF features to drop.")

print(f"\nFeatures after VIF filter: {len(final_features)}")

model = fit_logit_statsmodels(df, final_features, "default_flag_12m")
print("\n" + str(model.summary()))

E:\Study\rl-debt-collection\.venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


VIF (top 15 by VIF):
                          feature  vif
                 age_x_n_children  inf
        synth_int_age_x_nchildren  inf
    synth_seg_app_nom_gender_top2  inf
               mean_age_by_gender  inf
                   app_nom_gender  inf
                  count_by_gender  inf
            mean_income_by_gender  inf
    synth_seg_app_nom_gender_top1  inf
                 act_age_decile_1  inf
        synth_seg_prime_age_30_50  inf
              app_nom_home_status  inf
          mean_age_by_home_status  inf
       mean_income_by_home_status  inf
app_number_of_children_quartile_2  inf
  app_number_of_children_decile_1  inf

Dropping 31 high-VIF features:
  - age_x_n_children
  - synth_int_age_x_nchildren
  - synth_seg_app_nom_gender_top2
  - mean_age_by_gender
  - app_nom_gender
  - count_by_gender
  - mean_income_by_gender
  - synth_seg_app_nom_gender_top1
  - act_age_decile_1
  - synth_seg_prime_age_30_50
  - app_nom_home_status
  - mean_age_by_home_status
  - mean_inco


                           Logit Regression Results                           
Dep. Variable:       default_flag_12m   No. Observations:               389525
Model:                          Logit   Df Residuals:                   389491
Method:                           MLE   Df Model:                           33
Date:                Thu, 07 May 2026   Pseudo R-squ.:                  0.1618
Time:                        14:23:37   Log-Likelihood:                -27225.
converged:                       True   LL-Null:                       -32483.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------
const                                     5.4647      0.312     17.501      0.000       4.853       6.077
app_nom_job_code                         -1.1058 

In [9]:
significant = model.pvalues[model.pvalues < 0.05].index.tolist()
final_features = [f for f in significant if f != "const"]
print(f"Final feature set after p<0.05 filter: {len(final_features)} features")
for f in final_features:
    coef = model.params[f]
    pval = model.pvalues[f]
    print(f"  {f:<45}  coef={coef:+.5f}  p={pval:.4g}")

Final feature set after p<0.05 filter: 22 features
  app_nom_job_code                               coef=-1.10578  p=0
  synth_age_of_newest                            coef=-0.02282  p=7.402e-15
  synth_newest_account_months                    coef=-0.00415  p=0.0004052
  synth_avg_account_age                          coef=-0.00825  p=7.579e-05
  synth_oldest_mortgage_months                   coef=-0.00925  p=2.293e-11
  synth_seg_app_nom_job_code_top1                coef=+1.08448  p=8.534e-52
  synth_credit_age_years                         coef=-0.00949  p=1.477e-07
  marital_x_n_children                           coef=-0.42147  p=1.16e-82
  synth_int_inc_x_nchildren                      coef=-0.00022  p=3.628e-31
  synth_oldest_tradeline_v1                      coef=-0.00753  p=2.374e-06
  synth_oldest_card_months                       coef=-0.00455  p=0.0002744
  synth_n_late_90                                coef=-0.00753  p=3.051e-05
  synth_avg_account_age_months                

In [10]:
final_model = fit_logit_statsmodels(df, final_features, "default_flag_12m")
joblib.dump(final_model, OUTPUT_DIR / "final_model.pkl")
print(f"Saved: {OUTPUT_DIR / 'final_model.pkl'}")
print("\nFinal model summary:")
print(final_model.summary())

Saved: E:\Study\rl-debt-collection\.claude\worktrees\funny-dubinsky\artifacts\phase2_rerun_v2\final_model.pkl

Final model summary:


                           Logit Regression Results                           
Dep. Variable:       default_flag_12m   No. Observations:               389525
Model:                          Logit   Df Residuals:                   389502
Method:                           MLE   Df Model:                           22
Date:                Thu, 07 May 2026   Pseudo R-squ.:                  0.1617
Time:                        14:23:39   Log-Likelihood:                -27230.
converged:                       True   LL-Null:                       -32483.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------
const                                     5.1543      0.169     30.575      0.000       4.824       5.485
app_nom_job_code                         -1.1063  

## Performance Evaluation

Compute Gini, KS, Brier on train and OOT splits.

In [11]:
import statsmodels.api as sm
X_full = sm.add_constant(df[final_features].fillna(df[final_features].median(numeric_only=True)),
                         has_constant="add")
df["pd_score"] = final_model.predict(X_full)

train_mask = df["split"] == "train"
oot_mask   = df["split"] == "oot"

metrics = {
    "train": {
        "gini" : compute_gini(df.loc[train_mask, "default_flag_12m"], df.loc[train_mask, "pd_score"]),
        "ks"   : compute_ks  (df.loc[train_mask, "default_flag_12m"], df.loc[train_mask, "pd_score"]),
        "brier": compute_brier(df.loc[train_mask, "default_flag_12m"], df.loc[train_mask, "pd_score"]),
        "auc"  : (compute_gini(df.loc[train_mask, "default_flag_12m"], df.loc[train_mask, "pd_score"]) + 1) / 2,
    },
    "oot": {
        "gini" : compute_gini(df.loc[oot_mask, "default_flag_12m"], df.loc[oot_mask, "pd_score"]),
        "ks"   : compute_ks  (df.loc[oot_mask, "default_flag_12m"], df.loc[oot_mask, "pd_score"]),
        "brier": compute_brier(df.loc[oot_mask, "default_flag_12m"], df.loc[oot_mask, "pd_score"]),
        "auc"  : (compute_gini(df.loc[oot_mask, "default_flag_12m"], df.loc[oot_mask, "pd_score"]) + 1) / 2,
    },
}
print(json.dumps(metrics, indent=2))

{
  "train": {
    "gini": 0.6410536039484025,
    "ks": 0.48110621106021684,
    "brier": 0.015219356754642983,
    "auc": 0.8205268019742012
  },
  "oot": {
    "gini": 0.7181197604272356,
    "ks": 0.5475497513848578,
    "brier": 0.00815718374553534,
    "auc": 0.8590598802136178
  }
}


In [12]:
cal_train = compute_calibration_metrics(df.loc[train_mask, "default_flag_12m"], df.loc[train_mask, "pd_score"])
cal_oot   = compute_calibration_metrics(df.loc[oot_mask,   "default_flag_12m"], df.loc[oot_mask,   "pd_score"])
cal_df = pd.DataFrame({"train": cal_train, "oot": cal_oot})
print("Calibration metrics:")
print(cal_df.to_string())

Calibration metrics:
                      train            oot
o_to_e_bin1        1.007992       0.000000
o_to_e_bin2        0.995227       0.136039
o_to_e_bin3        0.946193       0.318722
o_to_e_bin4        0.953032       0.387455
o_to_e_bin5        0.990767       0.420720
o_to_e_bin6        0.858687       0.297431
o_to_e_bin7        0.923754       0.441476
o_to_e_bin8        0.983163       0.427902
o_to_e_bin9        1.014547       0.464239
o_to_e_bin10       1.034050       0.637332
ece                0.000622       0.007792
n             389525.000000  144789.000000
base_rate          0.016330       0.008474
mean_pred          0.016330       0.016267


## Safety Checks

Final-feature acceptance gate:
- No score/scorem under any name
- No `default_flag_12m` self-reference
- No loan-term transforms
- F6D pure-random count + tiered alert

In [13]:
# Check 1: no simulator score (synth_bureau_* allowed)
for f in final_features:
    fl = f.lower()
    if "score" in fl and "bureau" not in fl and "synth_int" not in fl:
        raise AssertionError(f"Score variable found: {f}")
print("Check 1 (no simulator score): PASS")

# Check 2: no target self-reference
assert "default_flag_12m" not in final_features
print("Check 2 (no target self-reference): PASS")

# Check 3: no loan-term transforms (look at raw string + catalog source_columns)
loan_terms = ("loan_amount", "n_installments", "installment", "principal", "tenor")
cat_idx = catalog.set_index("feature_name")
for f in final_features:
    fl = f.lower()
    # forbid raw-name match unless feature is a synthetic bureau-score derivative
    if f.startswith("synth_"):
        continue
    for lt in loan_terms:
        if lt in fl:
            raise AssertionError(f"Loan-term transform found: {f}")
    # cross-check against catalog source_columns
    if f in cat_idx.index:
        sources = str(cat_idx.at[f, "source_columns"] or "").lower()
        for lt in loan_terms:
            if lt in sources and not f.startswith("synth_"):
                raise AssertionError(f"Loan-term sourcing found in {f}: source_columns={sources}")
print("Check 3 (no loan-term transforms): PASS")

# Check 4: F6D negative-control assessment (tiered)
f6d_in_final = [f for f in final_features if f.startswith("random_")]
print(f"Check 4: F6D pure-random features surviving = {len(f6d_in_final)}")
if len(f6d_in_final) == 0:
    print("  IDEAL: zero F6D survived -> Lasso correctly rejects pure noise")
elif len(f6d_in_final) == 1:
    print(f"  WARNING: 1 F6D survived -> investigate but not auto-failure  -> {f6d_in_final[0]}")
else:
    print(f"  SERIOUS: {len(f6d_in_final)} F6D survived  -> investigate pipeline  -> {f6d_in_final}")

Check 1 (no simulator score): PASS
Check 2 (no target self-reference): PASS
Check 3 (no loan-term transforms): PASS
Check 4: F6D pure-random features surviving = 0
  IDEAL: zero F6D survived -> Lasso correctly rejects pure noise


## Save Artifacts

In [14]:
stage1.to_csv(OUTPUT_DIR / "stage1_selected_features.csv", index=False)
pd.Series(stage2, name="feature").to_csv(OUTPUT_DIR / "stage2_selected_features.csv", index=False, header=True)
pd.Series(final_features, name="feature").to_csv(OUTPUT_DIR / "final_feature_set.csv", index=False, header=True)

df_scores = df[["aid", "split", "default_flag_12m", "pd_score"]].copy()
df_scores.to_parquet(OUTPUT_DIR / "pd_scores.parquet", index=False)

with open(OUTPUT_DIR / "model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Persist final coefficient table
coef_table = pd.DataFrame({
    "feature": ["const"] + final_features,
    "coef": [final_model.params["const"]] + [final_model.params[f] for f in final_features],
    "std_err": [final_model.bse["const"]] + [final_model.bse[f] for f in final_features],
    "p_value": [final_model.pvalues["const"]] + [final_model.pvalues[f] for f in final_features],
})
coef_table.to_csv(OUTPUT_DIR / "final_coefficients.csv", index=False)

# Calibration outputs
cal_df.to_csv(OUTPUT_DIR / "calibration_metrics.csv")

print("Saved artifacts:")
for p in sorted(OUTPUT_DIR.glob("*")):
    sz = p.stat().st_size
    print(f"  {p.name:<40}  {sz:>10,} bytes")
print(f"\nWall (Phase 2 total): {time.time() - T0:.1f}s")

Saved artifacts:
  calibration_metrics.csv                          683 bytes
  final_coefficients.csv                         2,052 bytes
  final_feature_set.csv                            549 bytes
  final_model.pkl                           207,240,649 bytes
  model_metrics.json                               303 bytes
  modeling_abt.parquet                      2,381,776,565 bytes
  modeling_feature_catalog.csv                 480,190 bytes
  pd_scores.parquet                          6,925,995 bytes
  stage1_selected_features.csv                  10,076 bytes
  stage2_selected_features.csv                   1,580 bytes

Wall (Phase 2 total): 98.6s


## Final Report

| Metric | Train | OOT |
|--------|------:|----:|
| Gini   | see metrics dict above | |
| KS     | | |
| AUC    | | |
| Brier  | | |
| ECE    | see calibration metrics | |

See `model_metrics.json`, `calibration_metrics.csv`, and
`final_coefficients.csv` for the canonical numbers used by the thesis.

**F6D negative-control assessment:** see safety checks above.

**Comparison to old SET A':** SET A' (4 numeric + 5 categorical: `app_income`,
`app_number_of_children`, `app_spendings`, `act_age`, plus the five
categoricals) is the prior art baseline. The expected overlap with the new
final feature set is computed in the next cell.

In [15]:
SET_A_PRIME = {
    "app_income", "app_number_of_children", "app_spendings", "act_age",
    "app_nom_gender", "app_nom_marital_status", "app_nom_home_status",
    "app_nom_cars", "app_nom_job_code",
}
overlap = SET_A_PRIME & set(final_features)
new_only = set(final_features) - SET_A_PRIME
print(f"SET A' size: {len(SET_A_PRIME)}")
print(f"Final feature set size: {len(final_features)}")
print(f"Overlap with SET A': {len(overlap)}")
print(f"  overlap features: {sorted(overlap)}")
print(f"New-feature-factory features in final: {len(new_only)}")
print(f"  (showing first 30): {sorted(new_only)[:30]}")

SET A' size: 9
Final feature set size: 22
Overlap with SET A': 1
  overlap features: ['app_nom_job_code']
New-feature-factory features in final: 21
  (showing first 30): ['app_nom_branch', 'app_nom_city', 'marital_x_n_children', 'median_income_by_job_code', 'synth_age_of_newest', 'synth_avg_account_age', 'synth_avg_account_age_months', 'synth_credit_age_years', 'synth_int_inc_x_nchildren', 'synth_n_auto_lines', 'synth_n_card_lines', 'synth_n_installment_lines', 'synth_n_late_30', 'synth_n_late_90', 'synth_newest_account_months', 'synth_oldest_card_months', 'synth_oldest_installment_months', 'synth_oldest_mortgage_months', 'synth_oldest_tradeline_v1', 'synth_seg_app_nom_job_code_top1', 'synth_seg_app_nom_marital_status_top2']
